# Repaso 2: Datos semiestructurados
### Prueba de conocimientos y deducciones sobre documentos JSON sintéticos

**Realizado por:** Joaquín Bermeo

## Propósito de la actividad

Consolidar lo estudiado sobre datos semiestructurados trabajando con documentos JSON que simulan las órdenes de una tienda en línea. La actividad repasa la lectura de JSON, el reconocimiento de su estructura (claves opcionales, anidación, listas), el tratamiento de faltantes e inconsistencias de tipo y de formato, el aplanado hacia tablas, la validación con reglas y la construcción de resúmenes anidados, para terminar con una **deducción razonada** sobre la calidad de los datos y su transformación.

## Contenidos que se repasan

- Qué son los datos semiestructurados y cómo se diferencian de las tablas (notebooks de tipos de datos y de datos no estructurados, donde la API devuelve JSON).
- Operaciones sobre datos estructurados una vez aplanados: filtrado, limpieza, conversión de tipos, agregación (notebook de datos estructurados).
- Validación y serialización de resultados.

## Instrucciones

- Trabaje las secciones en orden: varios ejercicios reutilizan resultados o funciones de ejercicios anteriores.
- Registre cada resultado en la variable `respuesta_N` (donde N es el número del ejercicio), respetando exactamente el formato pedido.
- No modifique las celdas de preparación de datos; trabaje siempre sobre copias de los datos originales.
- Los ejercicios piden **cálculos y deducciones**, no gráficos.
- Los redondeos se indican en cada enunciado. Ante un empate se aplica la regla que se indique; si no se indica ninguna, no hay empates.
- Puntaje total de la prueba: **21 puntos**.

## Preparación de los datos

Los datos son **sintéticos**: simulan las órdenes de compra de una tienda en línea que el sistema entrega como un archivo JSON. La celda siguiente genera el archivo `ordenes_tienda.json` (siempre con el mismo contenido) y lo lee como texto.

In [1]:
import os, re, json, math, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
from datetime import datetime, timedelta
from collections import Counter


def generar_ordenes(n=400, seed=2026):
    rng = np.random.default_rng(seed)
    ciudades = {"Quito": "Pichincha", "Guayaquil": "Guayas", "Cuenca": "Azuay", "Ambato": "Tungurahua",
                "Loja": "Loja", "Riobamba": "Chimborazo", "Latacunga": "Cotopaxi", "Píllaro": "Tungurahua"}
    lista_ciudades = list(ciudades)
    catalogo = [("SKU-101", "Teclado mecánico", "Periféricos", 45.90), ("SKU-102", "Ratón inalámbrico", "Periféricos", 18.50),
                ("SKU-103", "Monitor 24 pulgadas", "Pantallas", 139.00), ("SKU-104", "Monitor 27 pulgadas", "Pantallas", 219.00),
                ("SKU-105", "Cable HDMI", "Accesorios", 7.80), ("SKU-106", "Hub USB", "Accesorios", 16.40),
                ("SKU-107", "Disco SSD 500 GB", "Almacenamiento", 54.90), ("SKU-108", "Memoria USB 64 GB", "Almacenamiento", 9.90),
                ("SKU-109", "Audífonos con micrófono", "Audio", 29.90), ("SKU-110", "Parlante portátil", "Audio", 35.00)]
    metodos = ["tarjeta", "transferencia", "efectivo"]
    pool = ["express", "regalo", "corporativo", "primera_compra"]
    empresas = ["Servientrega", "Tramaco", "Laar"]
    inicio = datetime(2025, 3, 1)
    ordenes = []
    for i in range(1, n + 1):
        f = inicio + timedelta(minutes=int(rng.integers(0, 60 * 24 * 120)))
        orden = {"id_orden": f"ORD-{i:05d}",
                 "fecha": f.strftime("%Y-%m-%dT%H:%M:%S") if rng.random() < .70 else f.strftime("%d/%m/%Y %H:%M")}
        cid = int(rng.integers(1, 121))
        ciudad = lista_ciudades[int(rng.integers(0, len(lista_ciudades)))]
        contacto = {"email": f"cliente{cid}@correo.ec"}
        u = rng.random()
        if u < .70:
            contacto["telefono"] = "09" + "".join(str(int(x)) for x in rng.integers(0, 10, 8))
        elif u < .80:
            contacto["telefono"] = None
        elif u < .90:
            contacto["telefono"] = ""
        cliente = {"id": cid, "nombre": f"Cliente {cid:03d}", "direccion": {"ciudad": ciudad, "provincia": ciudades[ciudad]}}
        if rng.random() > .05:
            cliente["contacto"] = contacto
        orden["cliente"] = cliente
        k = int(rng.integers(1, 5))
        items = []
        for j in rng.choice(len(catalogo), size=k, replace=False):
            sku, nom, cat, precio = catalogo[int(j)]
            items.append({"sku": sku, "nombre": nom, "categoria": cat,
                          "precio": precio if rng.random() > .04 else f"{precio:.2f}",
                          "cantidad": int(rng.integers(1, 4))})
        orden["items"] = items
        metodo = str(rng.choice(metodos, p=[.55, .30, .15]))
        pago = {"metodo": metodo, "estado": str(rng.choice(["aprobado", "pendiente", "rechazado"], p=[.80, .12, .08]))}
        if metodo == "tarjeta":
            pago["cuotas"] = int(rng.choice([1, 3, 6, 12]))
        orden["pago"] = pago
        v = rng.random()
        if v < .45:
            orden["etiquetas"] = sorted(str(x) for x in rng.choice(pool, size=int(rng.integers(1, 3)), replace=False))
        elif v < .60:
            orden["etiquetas"] = []
        v = rng.random()
        if v < .75:
            orden["envio"] = {"empresa": str(rng.choice(empresas)), "costo": round(float(rng.uniform(2, 8)), 2), "dias": int(rng.integers(1, 6))}
        elif v < .85:
            orden["envio"] = None
        if rng.random() < .05:
            orden["notas"] = "cliente solicita factura"
    # (se agregan al final: ordenes con defectos de calidad)
        ordenes.append(orden)
    for pos, idx in enumerate(rng.choice(n, size=12, replace=False)):
        o = ordenes[int(idx)]
        tipo = pos % 5
        if tipo == 0:
            o["items"] = []
        elif tipo == 1:
            del o["cliente"]["id"]
        elif tipo == 2:
            del o["pago"]["metodo"]
        elif tipo == 3:
            o["items"][0]["cantidad"] = 0
        else:
            o["items"][0]["precio"] = -10.0
    return ordenes


ARCHIVO_JSON = "ordenes_tienda.json"
with open(ARCHIVO_JSON, "w", encoding="utf-8") as f:
    json.dump(generar_ordenes(), f, ensure_ascii=False, indent=1)

with open(ARCHIVO_JSON, encoding="utf-8") as f:
    texto_json = f.read()
print("Archivo:", ARCHIVO_JSON, "| caracteres:", len(texto_json))
print(texto_json[:600])

Archivo: ordenes_tienda.json | caracteres: 315636
[
 {
  "id_orden": "ORD-00001",
  "fecha": "2025-06-11T05:20:00",
  "cliente": {
   "id": 22,
   "nombre": "Cliente 022",
   "direccion": {
    "ciudad": "Cuenca",
    "provincia": "Azuay"
   },
   "contacto": {
    "email": "cliente22@correo.ec",
    "telefono": "0946387797"
   }
  },
  "items": [
   {
    "sku": "SKU-101",
    "nombre": "Teclado mecánico",
    "categoria": "Periféricos",
    "precio": 45.9,
    "cantidad": 1
   }
  ],
  "pago": {
   "metodo": "efectivo",
   "estado": "aprobado"
  },
  "envio": {
   "empresa": "Tramaco",
   "costo": 4.69,
   "dias": 5
  }
 },
 {
  "id_orden":


La variable `texto_json` contiene el contenido **crudo** del archivo (una cadena de texto). A partir de ella trabajará en todos los ejercicios.

# Ejercicios

## Ejercicio 1. Lectura y estructura del documento (1 punto)

**Propósito.** Comprobar que sabe convertir texto JSON en estructuras de Python y reconocer que, a diferencia de una tabla, los documentos semiestructurados no tienen todos las mismas claves.

**Enunciado.** Convierta `texto_json` en objetos de Python. Determine:

- cuántas órdenes hay,
- qué claves de primer nivel aparecen en **todas** las órdenes,
- qué claves de primer nivel aparecen solo en **algunas**.

**Formato de respuesta:** diccionario con `n_ordenes` (int), `claves_en_todas` (lista ordenada alfabéticamente) y `claves_parciales` (lista ordenada alfabéticamente).

In [3]:
# Escriba su solución
RESP = {}
def registrar(n, valor):
    RESP[n] = valor
    print(json.dumps(valor, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))

ordenes = json.loads(texto_json)

n_ordenes = len(ordenes)
claves_en_todas = set(ordenes[0].keys())
todas_claves = set()

for o in ordenes:
    claves_en_todas.intersection_update(o.keys())
    todas_claves.update(o.keys())

claves_parciales = todas_claves - claves_en_todas

respuesta_1 = {
    "n_ordenes": int(n_ordenes),
    "claves_en_todas": sorted(list(claves_en_todas)),
    "claves_parciales": sorted(list(claves_parciales))
}

registrar(1, respuesta_1)

{
 "n_ordenes": 400,
 "claves_en_todas": [
  "cliente",
  "fecha",
  "id_orden",
  "items",
  "pago"
 ],
 "claves_parciales": [
  "envio",
  "etiquetas",
  "notas"
 ]
}


## Ejercicio 2. Profundidad y rutas del documento (2 puntos)

**Propósito.** Evaluar la comprensión de la anidación: un JSON puede tener varios niveles y su estructura se describe mediante rutas de acceso.

**Enunciado.** **Profundidad** de un documento: número máximo de diccionarios anidados (el documento raíz cuenta como nivel 1; entrar a un diccionario dentro de una lista también suma un nivel).

**Ruta hoja:** camino de claves, separadas por punto, hasta un valor terminal. Los elementos de las listas **no vacías** se indican con `[]` (por ejemplo `items[].precio`). Un valor simple, `null`, una lista vacía o un diccionario vacío terminan la ruta en ese punto.

Calcule la profundidad máxima entre todas las órdenes y el número de rutas hoja **distintas** que existen en el conjunto completo.

**Formato de respuesta:** diccionario con `profundidad_maxima` (int) y `rutas_hoja_distintas` (int).

In [4]:
# Escriba su solución
def calcular_profundidad(obj):
    if isinstance(obj, dict):
        if not obj:
            return 1
        return 1 + max([calcular_profundidad(v) for v in obj.values()] + [0])
    elif isinstance(obj, list):
        if not obj:
            return 0
        return max([calcular_profundidad(elem) for elem in obj] + [0])
    return 0

def obtener_rutas_hoja(obj, prefijo=""):
    rutas = set()
    if isinstance(obj, dict):
        if not obj:
            if prefijo:
                rutas.add(prefijo.rstrip("."))
        else:
            for k, v in obj.items():
                p = f"{prefijo}{k}"
                if isinstance(v, dict):
                    if not v:
                        rutas.add(p)
                    else:
                        rutas.update(obtener_rutas_hoja(v, p + "."))
                elif isinstance(v, list):
                    if not v:
                        rutas.add(p)
                    else:
                        for elem in v:
                            if isinstance(elem, dict):
                                rutas.update(obtener_rutas_hoja(elem, p + "[]."))
                            elif isinstance(elem, list):
                                rutas.update(obtener_rutas_hoja(elem, p + "[]."))
                            else:
                                rutas.add(p + "[]")
                else:
                    rutas.add(p)
    return rutas

profundidad_maxima = max(calcular_profundidad(o) for o in ordenes)

todas_rutas = set()
for o in ordenes:
    todas_rutas.update(obtener_rutas_hoja(o))

respuesta_2 = {
    "profundidad_maxima": int(profundidad_maxima),
    "rutas_hoja_distintas": int(len(todas_rutas))
}

registrar(2, respuesta_2)

{
 "profundidad_maxima": 3,
 "rutas_hoja_distintas": 24
}


## Ejercicio 3. Datos faltantes en documentos (2 puntos)

**Propósito.** Evaluar el manejo de campos ausentes, nulos y vacíos, que en JSON son tres situaciones distintas que una tabla suele confundir.

**Enunciado.** Cuente:

- las órdenes **sin teléfono** de contacto: cuando falta alguna clave en el camino `cliente.contacto.telefono`, el valor es `null` o es una cadena vacía (o solo espacios),
- las órdenes **sin envío**: la clave `envio` no existe o es `null`,
- las órdenes cuyo cliente **no tiene el campo `contacto`**.

**Formato de respuesta:** diccionario con `sin_telefono`, `sin_envio` y `cliente_sin_contacto` (todos int).

In [5]:
# Escriba su solución
sin_telefono = 0
sin_envio = 0
cliente_sin_contacto = 0

for o in ordenes:
    if "envio" not in o or o["envio"] is None:
        sin_envio += 1
    
    cliente = o.get("cliente", {})
    if not isinstance(cliente, dict) or "contacto" not in cliente or cliente["contacto"] is None:
        cliente_sin_contacto += 1
        sin_telefono += 1
    else:
        contacto = cliente["contacto"]
        tel = contacto.get("telefono") if isinstance(contacto, dict) else None
        if tel is None or (isinstance(tel, str) and tel.strip() == ""):
            sin_telefono += 1

respuesta_3 = {
    "sin_telefono": int(sin_telefono),
    "sin_envio": int(sin_envio),
    "cliente_sin_contacto": int(cliente_sin_contacto)
}

registrar(3, respuesta_3)

{
 "sin_telefono": 135,
 "sin_envio": 92,
 "cliente_sin_contacto": 20
}


## Ejercicio 4. Tipos inconsistentes (1 punto)

**Propósito.** Evaluar la detección y corrección de inconsistencias de tipo, frecuentes cuando los datos vienen de sistemas distintos.

**Enunciado.** Considere **todos** los ítems de todas las órdenes. Algunos precios llegan como texto en lugar de número.

Determine el total de ítems, cuántos precios llegan como texto y el precio medio por ítem tras convertir todos los precios a número (2 decimales).

**Formato de respuesta:** diccionario con `n_items` (int), `precios_texto` (int) y `precio_medio` (float).

In [6]:
# Escriba su solución
n_items = 0
precios_texto = 0
suma_precios = 0.0

for o in ordenes:
    for it in o.get("items", []):
        n_items += 1
        precio = it.get("precio")
        if isinstance(precio, str):
            precios_texto += 1
        suma_precios += float(precio)

precio_medio = round(suma_precios / n_items, 2)

respuesta_4 = {
    "n_items": int(n_items),
    "precios_texto": int(precios_texto),
    "precio_medio": float(precio_medio)
}

registrar(4, respuesta_4)

{
 "n_items": 975,
 "precios_texto": 36,
 "precio_medio": 60.97
}


## Ejercicio 5. Normalización de fechas (2 puntos)

**Propósito.** Evaluar la normalización de fechas que llegan en formatos distintos, paso imprescindible antes de cualquier análisis temporal.

**Enunciado.** El campo `fecha` llega en dos formatos: ISO (`2025-03-14T10:22:00`) y latino (`14/03/2025 10:22`). Convierta todas las fechas a un mismo tipo y determine:

- cuántas órdenes usan cada formato,
- el mes (`AAAA-MM`) con más órdenes y cuántas tiene,
- la fecha más antigua y la más reciente (formato `AAAA-MM-DD HH:MM`).

**Formato de respuesta:** diccionario con `n_iso` (int), `n_latino` (int), `mes_mas_ordenes` (str), `ordenes_mes` (int), `fecha_min` (str) y `fecha_max` (str).

In [7]:
# Escriba su solución
n_iso = 0
n_latino = 0
fechas = []

for o in ordenes:
    cadena_fecha = o["fecha"]
    if "T" in cadena_fecha:
        n_iso += 1
        dt = datetime.strptime(cadena_fecha, "%Y-%m-%dT%H:%M:%S")
    else:
        n_latino += 1
        dt = datetime.strptime(cadena_fecha, "%d/%m/%Y %H:%M")
    fechas.append(dt)

meses = [dt.strftime("%Y-%m") for dt in fechas]
conteo_meses = Counter(meses)
mes_mas_ordenes, ordenes_mes = conteo_meses.most_common(1)[0]

fecha_min = min(fechas).strftime("%Y-%m-%d %H:%M")
fecha_max = max(fechas).strftime("%Y-%m-%d %H:%M")

respuesta_5 = {
    "n_iso": int(n_iso),
    "n_latino": int(n_latino),
    "mes_mas_ordenes": str(mes_mas_ordenes),
    "ordenes_mes": int(ordenes_mes),
    "fecha_min": str(fecha_min),
    "fecha_max": str(fecha_max)
}

registrar(5, respuesta_5)

{
 "n_iso": 273,
 "n_latino": 127,
 "mes_mas_ordenes": "2025-03",
 "ordenes_mes": 107,
 "fecha_min": "2025-03-01 05:58",
 "fecha_max": "2025-06-28 23:12"
}


## Ejercicio 6. Aplanado de listas anidadas (2 puntos)

**Propósito.** Evaluar la conversión de una estructura anidada (lista de ítems dentro de cada orden) a una tabla con una fila por ítem, paso habitual para analizar JSON con herramientas de datos estructurados.

**Enunciado.** Construya una tabla con **una fila por ítem** (conservando `id_orden`, `sku`, `categoria`, `cantidad` y el precio convertido a número). Defina `ingreso = precio x cantidad`.

Determine el número de filas, las unidades totales vendidas, la categoría con mayor ingreso y su ingreso (2 decimales).

**Formato de respuesta:** diccionario con `filas_items` (int), `unidades` (int), `categoria_top` (str) y `ingreso_categoria_top` (float).

In [8]:
# Escriba su solución
filas_items = []

for o in ordenes:
    for it in o.get("items", []):
        filas_items.append({
            "id_orden": o["id_orden"],
            "sku": it["sku"],
            "categoria": it["categoria"],
            "cantidad": it["cantidad"],
            "precio": float(it["precio"])
        })

df_items = pd.DataFrame(filas_items)
df_items["ingreso"] = df_items["precio"] * df_items["cantidad"]

unidades = int(df_items["cantidad"].sum())
ingreso_por_cat = df_items.groupby("categoria")["ingreso"].sum()
categoria_top = str(ingreso_por_cat.idxmax())
ingreso_categoria_top = round(float(ingreso_por_cat.max()), 2)

respuesta_6 = {
    "filas_items": int(len(df_items)),
    "unidades": unidades,
    "categoria_top": categoria_top,
    "ingreso_categoria_top": ingreso_categoria_top
}

registrar(6, respuesta_6)

{
 "filas_items": 975,
 "unidades": 1948,
 "categoria_top": "Pantallas",
 "ingreso_categoria_top": 80856.0
}


## Ejercicio 7. Combinar niveles del documento (2 puntos)

**Propósito.** Evaluar la combinación de información de niveles distintos del documento: datos del cliente (nivel superior) con datos de los ítems (lista anidada).

**Enunciado.** Calcule el ingreso total de los ítems (`precio x cantidad`) agrupado por la **ciudad del cliente** (`cliente.direccion.ciudad`).

**Formato de respuesta:** diccionario con `ciudad` (la de mayor ingreso, str), `ingreso` (2 decimales) y `ordenes` (cantidad de órdenes de esa ciudad, int).

In [9]:
# Escriba su solución
ciudades_ingreso = {}
ciudades_ordenes = {}

for o in ordenes:
    ciudad = o["cliente"]["direccion"]["ciudad"]
    ciudades_ordenes[ciudad] = ciudades_ordenes.get(ciudad, 0) + 1
    
    ingreso_orden = sum(float(it["precio"]) * it["cantidad"] for it in o.get("items", []))
    ciudades_ingreso[ciudad] = ciudades_ingreso.get(ciudad, 0.0) + ingreso_orden

ciudad_top = max(ciudades_ingreso, key=ciudades_ingreso.get)

respuesta_7 = {
    "ciudad": str(ciudad_top),
    "ingreso": round(float(ciudades_ingreso[ciudad_top]), 2),
    "ordenes": int(ciudades_ordenes[ciudad_top])
}

registrar(7, respuesta_7)

{
 "ciudad": "Riobamba",
 "ingreso": 20028.5,
 "ordenes": 48
}


## Ejercicio 8. Listas de valores (1 punto)

**Propósito.** Evaluar el análisis de campos de tipo lista (etiquetas), que no existen en una tabla clásica.

**Enunciado.** Considere solo las órdenes cuyo campo `etiquetas` **existe y tiene al menos un elemento**. Determine cuántas son y las **3 etiquetas más frecuentes** con su frecuencia, de mayor a menor (ante empate, orden alfabético).

**Formato de respuesta:** diccionario con `ordenes_con_etiquetas` (int) y `top3` (lista de pares `[etiqueta, frecuencia]`).

In [10]:
# Escriba su solución
ordenes_con_etiquetas = 0
todas_etiquetas = []

for o in ordenes:
    etiquetas = o.get("etiquetas")
    if isinstance(etiquetas, list) and len(etiquetas) > 0:
        ordenes_con_etiquetas += 1
        todas_etiquetas.extend(etiquetas)

conteo_etiquetas = Counter(todas_etiquetas)
ordenadas = sorted(conteo_etiquetas.items(), key=lambda x: (-x[1], x[0]))
top3 = [[str(etiqueta), int(frec)] for etiqueta, frec in ordenadas[:3]]

respuesta_8 = {
    "ordenes_con_etiquetas": int(ordenes_con_etiquetas),
    "top3": top3
}

registrar(8, respuesta_8)

{
 "ordenes_con_etiquetas": 173,
 "top3": [
  [
   "primera_compra",
   76
  ],
  [
   "express",
   70
  ],
  [
   "corporativo",
   65
  ]
 ]
}


## Ejercicio 9. Validación de esquema (3 puntos)

**Propósito.** Evaluar la capacidad de definir y aplicar reglas de calidad sobre documentos, una tarea central al recibir datos semiestructurados de fuentes externas.

**Enunciado.** Una orden es **válida** si cumple **todas** estas reglas:

1. tiene `id_orden`,
2. su `cliente` tiene `id`,
3. `items` es una lista **no vacía**,
4. su `pago` tiene `metodo`,
5. todos sus ítems tienen precio (convertido a número) **mayor que 0** y cantidad **mayor o igual a 1**.

Determine cuántas órdenes son inválidas y los identificadores de las **3 primeras** (en orden alfabético).

**Formato de respuesta:** diccionario con `n_invalidas` (int) y `primeras_3` (lista de str).

In [11]:
# Escriba su solución
invalidas = []

for o in ordenes:
    oid = o.get("id_orden")
    if not oid:
        invalidas.append(oid)
        continue
    
    cliente = o.get("cliente")
    if not isinstance(cliente, dict) or "id" not in cliente or cliente["id"] is None:
        invalidas.append(oid)
        continue
    
    items = o.get("items")
    if not isinstance(items, list) or len(items) == 0:
        invalidas.append(oid)
        continue
    
    pago = o.get("pago")
    if not isinstance(pago, dict) or "metodo" not in pago or pago["metodo"] is None:
        invalidas.append(oid)
        continue
    
    items_validos = True
    for it in items:
        try:
            precio = float(it.get("precio", 0))
            cantidad = int(it.get("cantidad", 0))
            if precio <= 0 or cantidad < 1:
                items_validos = False
                break
        except (ValueError, TypeError):
            items_validos = False
            break
            
    if not items_validos:
        invalidas.append(oid)

primeras_3 = sorted(invalidas)[:3]

respuesta_9 = {
    "n_invalidas": int(len(invalidas)),
    "primeras_3": primeras_3
}

registrar(9, respuesta_9)

{
 "n_invalidas": 12,
 "primeras_3": [
  "ORD-00002",
  "ORD-00013",
  "ORD-00024"
 ]
}


## Ejercicio 10. Resumen anidado y serialización (2 puntos)

**Propósito.** Evaluar la construcción de un resumen con estructura anidada, que es lo que habitualmente se devuelve a otro sistema en formato JSON.

**Enunciado.** Agrupe las órdenes por `pago.metodo` (las órdenes sin método se agrupan bajo `sin_metodo`). Para cada grupo calcule: número de órdenes, ingreso de ítems (`precio x cantidad`, 2 decimales) y cuántas órdenes tienen `pago.estado` igual a `aprobado`.

**Formato de respuesta:** diccionario `{metodo: {"ordenes": int, "ingreso_items": float, "aprobadas": int}}`, que debe poder convertirse a JSON con `json.dumps`.

In [12]:
# Escriba su solución
resumen_pago = {}

for o in ordenes:
    pago = o.get("pago")
    metodo = pago.get("metodo") if isinstance(pago, dict) else None
    if not metodo:
        metodo = "sin_metodo"
        
    if metodo not in resumen_pago:
        resumen_pago[metodo] = {"ordenes": 0, "ingreso_items": 0.0, "aprobadas": 0}
        
    resumen_pago[metodo]["ordenes"] += 1
    
    ingreso_orden = sum(float(it["precio"]) * it["cantidad"] for it in o.get("items", []))
    resumen_pago[metodo]["ingreso_items"] += ingreso_orden
    
    if isinstance(pago, dict) and pago.get("estado") == "aprobado":
        resumen_pago[metodo]["aprobadas"] += 1

for m in resumen_pago:
    resumen_pago[m]["ingreso_items"] = round(float(resumen_pago[m]["ingreso_items"]), 2)

respuesta_10 = resumen_pago

registrar(10, respuesta_10)

{
 "efectivo": {
  "ordenes": 59,
  "ingreso_items": 16851.2,
  "aprobadas": 48
 },
 "transferencia": {
  "ordenes": 116,
  "ingreso_items": 31325.7,
  "aprobadas": 91
 },
 "tarjeta": {
  "ordenes": 223,
  "ingreso_items": 74167.8,
  "aprobadas": 179
 },
 "sin_metodo": {
  "ordenes": 2,
  "ingreso_items": 196.8,
  "aprobadas": 2
 }
}


## Ejercicio 11. Deducción razonada (3 puntos)

**Propósito.** Evaluar la capacidad de interpretar los resultados: identificar problemas de calidad, justificar por qué JSON es el formato adecuado y proponer una decisión para llevar los datos a una tabla.

**Enunciado.** Con base en los resultados de los ejercicios 3, 4, 5 y 9, redacte entre 5 y 8 líneas que expliquen:

- qué **problemas de calidad** presenta este conjunto de documentos,
- por qué un formato **semiestructurado** resulta adecuado para este tipo de dato,
- qué **decisión** tomaría para llevarlo a una tabla (qué aplanaría, qué descartaría, cómo trataría los faltantes).

Incluya al menos **tres cifras** obtenidas en sus cálculos.

**Formato de respuesta:** texto redactado en la celda de respuesta.

**Escriba aquí su respuesta:**

El conjunto de documentos presenta diversos problemas de calidad: heterogeneidad en el formato de fechas (273 órdenes en formato ISO y 127 en formato latino), inconsistencia de tipos (36 precios registrados como texto de un total de 975 ítems) y campos opcionales o faltantes (135 órdenes sin teléfono y 92 sin información de envío), además de 12 órdenes con esquemas inválidos por claves omitidas, cantidades en cero o precios negativos. A pesar de estas inconsistencias, el formato semiestructurado (JSON) es el más adecuado para este tipo de datos porque modela de manera flexible relaciones jerárquicas y colecciones de tamaño variable (múltiples ítems por orden, etiquetas y opciones de envío condicionales) sin forzar una estructura fija. Para migrar estos datos a tablas estructuradas, la mejor decisión es desacoplar el modelo en dos tablas relacionales vinculadas por id_orden: una tabla de órdenes con fechas normalizadas y valores faltantes tratados como nulos, y una tabla de detalle de ítems aplanada con precios tipificados a float, descartando o aislando para reproceso las 12 órdenes que no cumplen con las reglas mínimas del esquema.